# Notebook 04 - LightGBM Tweedie vs Gaussian on Intermittent Demand

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from retail_forecast.config import PROCESSED_DATA_DIR, MODELS_DIR, REPORTS_DIR
from retail_forecast.models.lgbm import LGBMTweedie, LGBMGaussian
from retail_forecast.evaluate import evaluate

FIGS_DIR = REPORTS_DIR / "nb04_tweedie_vs_gaussian"
FIGS_DIR.mkdir(parents=True, exist_ok=True)

CAT_COLORS = {"FOODS": "#636EFA", "HOBBIES": "#EF553B", "HOUSEHOLD": "#00CC96"}
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (12, 4)

In [ ]:
# Load feature matrix from cache (built in NB03)
feat_cache = PROCESSED_DATA_DIR / "feature_matrix.parquet"
if not feat_cache.exists():
    raise FileNotFoundError(
        "Feature matrix cache not found. Run Notebook 03 first to build it."
    )

feat = pd.read_parquet(feat_cache)

# Restore category dtype (parquet may lose it)
for col in ["cat_id", "dept_id"]:
    if col in feat.columns:
        feat[col] = feat[col].astype("category")

print(f"Feature matrix loaded: {feat.shape}")
print(f"Date range: {feat['date'].min().date()} -> {feat['date'].max().date()}")

## 1. Train Both Models

**Same feature set, same train/val/test split - only the objective function differs.**

- `LGBMTweedie`: `objective='tweedie'`, `tweedie_variance_power=1.1`
- `LGBMGaussian`: `objective='regression'` (L2 = Gaussian assumption)

Early stopping on validation set prevents overfitting in both cases.


In [ ]:
dates = feat["date"].sort_values().unique()
n = len(dates)
train_end = dates[int(n * 0.80)]
val_end   = dates[int(n * 0.90)]

train_df = feat[feat["date"] <= train_end]
val_df   = feat[(feat["date"] > train_end) & (feat["date"] <= val_end)]
test_df  = feat[feat["date"] > val_end]

print(f"Train: {len(train_df):,} rows  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}")

models = {}
for ModelClass, key in [(LGBMTweedie, "tweedie"), (LGBMGaussian, "gaussian")]:
    print(f"\nTraining {key}...")
    m = ModelClass(num_boost_round=1000, early_stopping_rounds=50)
    m.fit(train_df, val_df)
    models[key] = m
    print(f"  best iteration: {m._booster.best_iteration}")

## 2. Overall Test-Set Metrics

Evaluate both models on the 10% holdout test set.  
WMAPE is weighted - robust to zero-inflation.  
RMSE penalises large errors more heavily - sensitive to event-day spikes.


In [ ]:
results = {}
for key, m in models.items():
    preds = np.maximum(m.predict(test_df), 0)
    results[key] = {"preds": preds, "metrics": evaluate(test_df["sales"].values, preds)}

metrics_df = pd.DataFrame({k: v["metrics"] for k, v in results.items()}).T
metrics_df.index.name = "Model"

# Add absolute and relative improvement
for m in ["rmse", "mae", "wmape"]:
    tw = metrics_df.loc["tweedie", m]
    ga = metrics_df.loc["gaussian", m]
    metrics_df.loc["tweedie", f"{m}_vs_gaussian"] = (tw - ga) / ga * 100

print("Test set metrics:")
print(metrics_df[["rmse", "mae", "wmape"]].round(4).to_string())
print("\nTweedie improvement over Gaussian (%):")
imp_cols = [c for c in metrics_df.columns if "vs_gaussian" in c]
print(metrics_df.loc[["tweedie"], imp_cols].round(2).to_string())

In [ ]:
# Side-by-side bar chart
metric_long = []
for key in ["tweedie", "gaussian"]:
    for m in ["rmse", "mae", "wmape"]:
        metric_long.append({"Model": f"LGBM {key.capitalize()}",
                             "Metric": m.upper(),
                             "Value": results[key]["metrics"][m]})

fig = px.bar(
    pd.DataFrame(metric_long),
    x="Model", y="Value", color="Model",
    facet_col="Metric", facet_col_spacing=0.08,
    title="LGBMTweedie vs LGBMGaussian - Overall Test-Set Metrics",
    labels={"Value": "Metric value"},
    color_discrete_map={"LGBM Tweedie": "#00CC96", "LGBM Gaussian": "#FFA15A"},
    height=380,
)
fig.update_layout(showlegend=True)
fig.write_html(str(FIGS_DIR / "overall_metrics.html"), include_plotlyjs="cdn")
fig.show()

## 3. Category-Level Metric Comparison

**Key hypothesis:** Tweedie advantage is largest for HOBBIES (most intermittent, ~72% zeros in training)
and smallest for FOODS (least intermittent, ~56% zeros in training).
HOUSEHOLD sits in between at ~68% zeros.

In [ ]:
test_aug = test_df.copy()
test_aug["y_tweedie"]  = results["tweedie"]["preds"]
test_aug["y_gaussian"] = results["gaussian"]["preds"]

cat_rows = []
for cat, grp in test_aug.groupby("cat_id", observed=True):
    y = grp["sales"].values
    for key, pred_col in [("Tweedie", "y_tweedie"), ("Gaussian", "y_gaussian")]:
        m = evaluate(y, grp[pred_col].values)
        cat_rows.append({"Category": str(cat), "Model": key, **m})

cat_df = pd.DataFrame(cat_rows)
print("Metrics by category:")
print(cat_df.pivot(index="Category", columns="Model").round(4).to_string())

In [ ]:
fig = make_subplots(rows=1, cols=3,
    subplot_titles=["RMSE", "MAE", "WMAPE"], horizontal_spacing=0.1)

MODEL_COLORS = {"Tweedie": "#00CC96", "Gaussian": "#FFA15A"}
CATS = ["FOODS", "HOBBIES", "HOUSEHOLD"]
shown = set()

for col_idx, metric in enumerate(["rmse", "mae", "wmape"], start=1):
    for model_name, color in MODEL_COLORS.items():
        sub = cat_df[cat_df["Model"] == model_name]
        sub = sub.set_index("Category").reindex(CATS)
        show_leg = model_name not in shown
        shown.add(model_name)
        fig.add_trace(go.Bar(
            x=CATS, y=sub[metric].values,
            name=f"LGBM {model_name}", marker_color=color,
            showlegend=show_leg, legendgroup=model_name,
        ), row=1, col=col_idx)

fig.update_layout(
    title="Tweedie vs Gaussian: Metrics by Category",
    barmode="group", height=420,
    legend=dict(orientation="h", y=1.12, x=0.5, xanchor="center"),
)
fig.write_html(str(FIGS_DIR / "metrics_by_category.html"), include_plotlyjs="cdn")
fig.show()

## 4. WRMSSE by Aggregation Level

M5 multi-level evaluation: WRMSSE at item, department, category, and total store level.

**Key observation:** Tweedie's advantage over Gaussian grows as aggregation increases.
At the item level the gap is small (0.006); at total store level it widens to 0.079.
This occurs because Tweedie's lower bias on zero-demand days compounds across the hierarchy —
the systematic over-prediction of Gaussian accumulates into visible aggregate-level error.

In [ ]:
from retail_forecast.evaluate import wrmsse_by_level

level_results = {}
for key in ["tweedie", "gaussian"]:
    preds = np.maximum(models[key].predict(test_df), 0)
    level_results[key] = wrmsse_by_level(train_df, test_df, preds)

# Comparison table
print(f"{'Level':<12} {'Tweedie':>10} {'Gaussian':>10} {'Tweedie gain':>14}")
print("-" * 50)
for level in level_results["tweedie"]:
    tw = level_results["tweedie"][level]
    ga = level_results["gaussian"][level]
    print(f"{level:<12} {tw:>10.4f} {ga:>10.4f} {ga - tw:>+14.4f}")

# Grouped bar chart
levels = list(level_results["tweedie"].keys())
fig = go.Figure()
fig.add_trace(go.Bar(
    name="Tweedie",
    x=levels,
    y=[level_results["tweedie"][l] for l in levels],
    text=[f"{level_results['tweedie'][l]:.3f}" for l in levels],
    textposition="outside",
    marker_color="#00CC96",
))
fig.add_trace(go.Bar(
    name="Gaussian",
    x=levels,
    y=[level_results["gaussian"][l] for l in levels],
    text=[f"{level_results['gaussian'][l]:.3f}" for l in levels],
    textposition="outside",
    marker_color="#FFA15A",
))
fig.add_hline(
    y=1.0, line_dash="dash", line_color="gray",
    annotation_text="Naive baseline (WRMSSE = 1.0)",
    annotation_position="top right",
)
fig.update_layout(
    title="WRMSSE by Aggregation Level - Tweedie vs Gaussian",
    xaxis_title="Aggregation Level",
    yaxis_title="WRMSSE (lower is better)",
    yaxis=dict(range=[0, 1.3]),
    barmode="group",
    template="plotly_white",
    legend=dict(x=0.75, y=0.95),
    width=700, height=440,
)
fig.show()

## 4b. Actual vs Predicted — Hierarchy Levels

Time-series comparison of Tweedie vs Gaussian across four aggregation levels.
As items are aggregated, individual zero-demand noise averages out and predictions
track actuals more closely — reflected in sharply lower WMAPE at higher levels.
Gaussian's systematic over-prediction bias is most visible at the aggregated levels.

- **Item (3 examples):** One item per category near the 75th percentile of zero-inflation,
  where the Tweedie vs Gaussian gap is most pronounced
- **Category (3):** FOODS / HOBBIES / HOUSEHOLD daily totals
- **Department (7):** All 7 departments
- **Total:** Store-wide daily demand

In [ ]:
# Helper used across all aggregation-level charts
def _wmape(actual, pred):
    d = actual.abs().sum()
    return (actual - pred).abs().sum() / d if d > 0 else 0

# Item-level: pick one item per category near the 75th percentile of zero-inflation
# (highly intermittent, where the Tweedie vs Gaussian difference is most visible)
zero_by_item_test = (
    test_aug.groupby(["id", "cat_id"], observed=True)["sales"]
    .apply(lambda s: (s == 0).mean())
    .reset_index(name="zero_pct")
)

def pick_p75(cat):
    sub = zero_by_item_test[zero_by_item_test["cat_id"] == cat]
    target = sub["zero_pct"].quantile(0.75)
    return sub.iloc[(sub["zero_pct"] - target).abs().argsort().iloc[0]]["id"]

sample_items_n4 = {cat: pick_p75(cat) for cat in ["FOODS", "HOBBIES", "HOUSEHOLD"]}
for cat, item in sample_items_n4.items():
    z = zero_by_item_test.set_index("id").loc[item, "zero_pct"]
    print(f"  {cat}: {item}  (zero-inflation {z:.1%})")

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    subplot_titles=[f"{cat} — {sample_items_n4[cat]}" for cat in ["FOODS", "HOBBIES", "HOUSEHOLD"]])

for i, cat in enumerate(["FOODS", "HOBBIES", "HOUSEHOLD"], 1):
    item_id = sample_items_n4[cat]
    sub = test_aug[test_aug["id"] == item_id].sort_values("date")
    wt = _wmape(sub["sales"], sub["y_tweedie"])
    wg = _wmape(sub["sales"], sub["y_gaussian"])
    fig.layout.annotations[i - 1].text = (
        f"{item_id}  (Tweedie WMAPE {wt:.1%} / Gaussian {wg:.1%})"
    )
    fig.add_trace(go.Scatter(x=sub["date"], y=sub["sales"],
        mode="lines+markers", name="Actual",
        line=dict(color="grey", width=1), marker=dict(size=3),
        legendgroup="actual", showlegend=(i == 1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=sub["date"], y=sub["y_tweedie"],
        mode="lines", name="Tweedie",
        line=dict(color="#00CC96", width=2, dash="dot"),
        legendgroup="tweedie", showlegend=(i == 1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=sub["date"], y=sub["y_gaussian"],
        mode="lines", name="Gaussian",
        line=dict(color="#FFA15A", width=2, dash="dash"),
        legendgroup="gaussian", showlegend=(i == 1)), row=i, col=1)
    fig.update_yaxes(title_text="Units", row=i, col=1)

fig.update_layout(
    title="Actual vs Predicted — Item Level (high-intermittency examples)",
    height=320 * 3, template="plotly_white",
    legend=dict(orientation="h", y=1.03, x=0.5, xanchor="center"),
)
fig.update_xaxes(title_text="Date", row=3, col=1)
fig.write_html(str(FIGS_DIR / "avp_item.html"), include_plotlyjs="cdn")
fig.show()

In [ ]:
# Category-level: sum per (cat_id, date)
cat_order = ["FOODS", "HOBBIES", "HOUSEHOLD"]

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    subplot_titles=cat_order)

for i, cat in enumerate(cat_order, 1):
    agg = (
        test_aug[test_aug["cat_id"] == cat]
        .groupby("date")[["sales", "y_tweedie", "y_gaussian"]].sum().reset_index()
    )
    wt = _wmape(agg["sales"], agg["y_tweedie"])
    wg = _wmape(agg["sales"], agg["y_gaussian"])
    fig.layout.annotations[i - 1].text = f"{cat}  (Tweedie {wt:.1%} / Gaussian {wg:.1%})"
    fig.add_trace(go.Scatter(x=agg["date"], y=agg["sales"], mode="lines", name="Actual",
        line=dict(color="grey", width=1.5), legendgroup="actual", showlegend=(i == 1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=agg["date"], y=agg["y_tweedie"], mode="lines", name="Tweedie",
        line=dict(color="#00CC96", width=2, dash="dot"),
        legendgroup="tweedie", showlegend=(i == 1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=agg["date"], y=agg["y_gaussian"], mode="lines", name="Gaussian",
        line=dict(color="#FFA15A", width=2, dash="dash"),
        legendgroup="gaussian", showlegend=(i == 1)), row=i, col=1)
    fig.update_yaxes(title_text="Units/day", row=i, col=1)

fig.update_layout(
    title="Actual vs Predicted — Category Level",
    height=300 * 3, template="plotly_white",
    legend=dict(orientation="h", y=1.03, x=0.5, xanchor="center"),
)
fig.update_xaxes(title_text="Date", row=3, col=1)
fig.write_html(str(FIGS_DIR / "avp_cat.html"), include_plotlyjs="cdn")
fig.show()

In [ ]:
# Department-level: sum per (dept_id, date)
DEPT_COLORS_N4 = {
    "FOODS_1": "#636EFA", "FOODS_2": "#19D3F3", "FOODS_3": "#00CC96",
    "HOBBIES_1": "#EF553B", "HOBBIES_2": "#FFA15A",
    "HOUSEHOLD_1": "#AB63FA", "HOUSEHOLD_2": "#FF6692",
}
dept_order = ["FOODS_1", "FOODS_2", "FOODS_3", "HOBBIES_1", "HOBBIES_2", "HOUSEHOLD_1", "HOUSEHOLD_2"]

fig = make_subplots(rows=7, cols=1, shared_xaxes=True, vertical_spacing=0.04,
    subplot_titles=dept_order)

for i, dept in enumerate(dept_order, 1):
    agg = (
        test_aug[test_aug["dept_id"] == dept]
        .groupby("date")[["sales", "y_tweedie", "y_gaussian"]].sum().reset_index()
    )
    wt = _wmape(agg["sales"], agg["y_tweedie"])
    wg = _wmape(agg["sales"], agg["y_gaussian"])
    fig.layout.annotations[i - 1].text = f"{dept}  (Tweedie {wt:.1%} / Gaussian {wg:.1%})"
    color = DEPT_COLORS_N4.get(dept, "#636EFA")
    fig.add_trace(go.Scatter(x=agg["date"], y=agg["sales"], mode="lines", name="Actual",
        line=dict(color="grey", width=1), legendgroup="actual", showlegend=(i == 1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=agg["date"], y=agg["y_tweedie"], mode="lines", name="Tweedie",
        line=dict(color="#00CC96", width=1.8, dash="dot"),
        legendgroup="tweedie", showlegend=(i == 1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=agg["date"], y=agg["y_gaussian"], mode="lines", name="Gaussian",
        line=dict(color="#FFA15A", width=1.8, dash="dash"),
        legendgroup="gaussian", showlegend=(i == 1)), row=i, col=1)
    fig.update_yaxes(title_text="Units/day", row=i, col=1)

fig.update_layout(
    title="Actual vs Predicted — Department Level",
    height=280 * 7, template="plotly_white",
    legend=dict(orientation="h", y=1.01, x=0.5, xanchor="center"),
)
fig.update_xaxes(title_text="Date", row=7, col=1)
fig.write_html(str(FIGS_DIR / "avp_dept.html"), include_plotlyjs="cdn")
fig.show()

In [ ]:
# Total: store-wide daily demand
agg_total = (
    test_aug.groupby("date")[["sales", "y_tweedie", "y_gaussian"]]
    .sum().reset_index()
)

wt = _wmape(agg_total["sales"], agg_total["y_tweedie"])
wg = _wmape(agg_total["sales"], agg_total["y_gaussian"])

fig = go.Figure()
fig.add_trace(go.Scatter(x=agg_total["date"], y=agg_total["sales"],
    mode="lines", name="Actual", line=dict(color="grey", width=2)))
fig.add_trace(go.Scatter(x=agg_total["date"], y=agg_total["y_tweedie"],
    mode="lines", name=f"Tweedie  (WMAPE {wt:.1%})",
    line=dict(color="#00CC96", width=2, dash="dot")))
fig.add_trace(go.Scatter(x=agg_total["date"], y=agg_total["y_gaussian"],
    mode="lines", name=f"Gaussian  (WMAPE {wg:.1%})",
    line=dict(color="#FFA15A", width=2, dash="dash")))
fig.update_layout(
    title="Actual vs Predicted — Total Store",
    xaxis_title="Date", yaxis_title="Total units sold per day",
    height=420, template="plotly_white",
    legend=dict(orientation="h", y=1.05),
)
fig.write_html(str(FIGS_DIR / "avp_total.html"), include_plotlyjs="cdn")
fig.show()

## 5. Zero-Demand Day Analysis

On days where **actual sales = 0**, how much does each model over-predict?
Gaussian (L2) minimises squared error symmetrically — it is pulled toward the mean,
which is positive. Tweedie's compound Poisson-Gamma structure puts more mass at zero,
so it predicts lower values on zero-demand days.

In the test set, **54.8%** of all item-day rows have zero actual sales.
On those days, Tweedie's mean prediction (0.5759) is measurably lower than Gaussian's (0.5970),
confirming the structural advantage of the Tweedie objective for intermittent demand.

In [ ]:
zero_days = test_aug[test_aug["sales"] == 0]
nonzero_days = test_aug[test_aug["sales"] > 0]

print(f"Zero-demand days in test: {len(zero_days):,}  ({len(zero_days)/len(test_aug):.1%})")
print(f"Non-zero days          : {len(nonzero_days):,}")
print()

for subset_name, subset in [("Zero-demand days (actual=0)", zero_days),
                             ("Non-zero days (actual>0)",  nonzero_days)]:
    print(f"--- {subset_name} ---")
    for key in ["tweedie", "gaussian"]:
        col = f"y_{key}"
        print(f"  {key.capitalize()}: mean_pred={subset[col].mean():.4f}  ",
              f"median_pred={subset[col].median():.4f}")
    print()

In [ ]:
# Bar chart: avg prediction on zero-demand days by model and category
zero_rows = []
for cat, grp in zero_days.groupby("cat_id", observed=True):
    for key in ["tweedie", "gaussian"]:
        zero_rows.append({
            "Category": str(cat),
            "Model": f"LGBM {key.capitalize()}",
            "Avg prediction on zero days": grp[f"y_{key}"].mean(),
        })

zero_comp = pd.DataFrame(zero_rows)

fig = px.bar(
    zero_comp,
    x="Category", y="Avg prediction on zero days",
    color="Model", barmode="group",
    title="Average Prediction on Zero-Demand Days by Category\n(lower = better for intermittent demand)",
    labels={"Avg prediction on zero days": "Avg. predicted units (true = 0)"},
    color_discrete_map={"LGBM Tweedie": "#00CC96", "LGBM Gaussian": "#FFA15A"},
    height=380,
)
fig.write_html(str(FIGS_DIR / "zero_demand_predictions.html"), include_plotlyjs="cdn")
fig.show()

## 6. Residual Distributions

Residuals = actual - predicted.
- **Gaussian bias:** On intermittent items, Gaussian over-predicts zeros — residuals skewed negative.
- **Tweedie:** Residuals more centred around zero, especially in HOBBIES (most zero-inflated).

In [ ]:
test_aug["resid_tweedie"]  = test_aug["sales"] - test_aug["y_tweedie"]
test_aug["resid_gaussian"] = test_aug["sales"] - test_aug["y_gaussian"]

# Sample for plot performance
sample = test_aug.sample(n=min(80_000, len(test_aug)), random_state=42)

fig = make_subplots(rows=1, cols=3,
    subplot_titles=["FOODS", "HOBBIES", "HOUSEHOLD"],
    horizontal_spacing=0.08)

shown_resid = set()
for col_idx, cat in enumerate(["FOODS", "HOBBIES", "HOUSEHOLD"], start=1):
    sub = sample[sample["cat_id"] == cat]
    for key, color, dash in [("tweedie", "#00CC96", "solid"),
                              ("gaussian", "#FFA15A", "dash")]:
        resid = sub[f"resid_{key}"].clip(-10, 10)
        show_leg = key not in shown_resid
        shown_resid.add(key)
        fig.add_trace(go.Histogram(
            x=resid, nbinsx=60, opacity=0.55,
            name=f"LGBM {key.capitalize()}",
            marker_color=color,
            legendgroup=key, showlegend=show_leg,
        ), row=1, col=col_idx)
    fig.add_vline(x=0, row=1, col=col_idx,
                  line_dash="dot", line_color="black", line_width=1)

fig.update_layout(
    title="Residual Distributions by Category (clipped to +-10, actual − predicted)",
    barmode="overlay", height=420,
    legend=dict(orientation="h", y=1.12, x=0.5, xanchor="center"),
)
fig.update_xaxes(title_text="Residual (actual − predicted)")
fig.write_html(str(FIGS_DIR / "residual_distributions.html"), include_plotlyjs="cdn")
fig.show()

## 7. Intermittency vs Tweedie Advantage

**The core novelty result:**
For each item, compute its zero-inflation rate (from the full training set) and its
RMSE improvement when switching from Gaussian to Tweedie.

Expected pattern: higher zero-inflation → bigger Tweedie gain
(positive correlation between x=zero_pct and y=RMSE_gaussian - RMSE_tweedie)

Median RMSE improvement (%) confirms the hypothesis:
- **HOBBIES** (most intermittent, ~72% zeros): **+0.79%**
- **HOUSEHOLD** (~68% zeros): **+0.57%**
- **FOODS** (least intermittent, ~56% zeros): **+0.51%**

In [ ]:
# Per-item zero-inflation rate (on training set)
zero_by_item = (
    feat[feat["date"] <= train_end]
    .groupby(["id", "cat_id"])["sales"]
    .apply(lambda s: (s == 0).mean())
    .reset_index(name="zero_pct")
)

# Per-item RMSE for both models on test set
item_rows = []
for item_id, grp in test_aug.groupby("id"):
    y = grp["sales"].values
    if len(y) < 5:
        continue
    rmse_tw = evaluate(y, grp["y_tweedie"].values)["rmse"]
    rmse_ga = evaluate(y, grp["y_gaussian"].values)["rmse"]
    cat = grp["cat_id"].iloc[0]
    item_rows.append({"id": item_id, "cat_id": str(cat),
                      "rmse_tweedie": rmse_tw, "rmse_gaussian": rmse_ga})

item_df = pd.DataFrame(item_rows)
item_df = item_df.merge(zero_by_item[["id", "zero_pct"]], on="id", how="left")
item_df["rmse_improvement"] = item_df["rmse_gaussian"] - item_df["rmse_tweedie"]
item_df["rmse_improvement_pct"] = item_df["rmse_improvement"] / item_df["rmse_gaussian"].replace(0, np.nan) * 100

print(f"Items analysed: {len(item_df):,}")
print("\nMedian RMSE improvement (Gaussian - Tweedie) by category:")
print(item_df.groupby("cat_id")["rmse_improvement_pct"].median().round(2).to_string())

In [ ]:
# Scatter: zero-inflation rate vs RMSE improvement (Tweedie gain)
# Downsample per category - avoid groupby/apply which drops cat_id in some pandas versions
plot_parts = []
for cat in item_df["cat_id"].unique():
    g = item_df[item_df["cat_id"] == cat]
    plot_parts.append(g.sample(min(len(g), 2000), random_state=42))
plot_df = pd.concat(plot_parts, ignore_index=True)

# Clip extreme outliers for readability
plot_df["rmse_improvement_pct"] = plot_df["rmse_improvement_pct"].clip(-50, 100)

fig = px.scatter(
    plot_df,
    x="zero_pct", y="rmse_improvement_pct",
    color="cat_id", color_discrete_map=CAT_COLORS,
    facet_col="cat_id",
    opacity=0.35,
    trendline="lowess",
    title="Tweedie Advantage vs Item Zero-Inflation Rate",
    labels={
        "zero_pct": "Item zero-inflation rate",
        "rmse_improvement_pct": "RMSE improvement: Gaussian − Tweedie (%)",
    },
    height=420,
)
fig.add_hline(y=0, line_dash="dash", line_color="red", line_width=1,
              annotation_text="Gaussian = Tweedie", annotation_position="top right")
fig.update_layout(showlegend=False)
fig.write_html(str(FIGS_DIR / "intermittency_vs_advantage.html"), include_plotlyjs="cdn")
fig.show()

## 8. Feature Importance Comparison

Both models learn from the same 37 features; only the objective function differs.
Differences in relative importance reveal what each objective prioritises.

Key observations from the SHAP analysis:
- **Tweedie** ranks `sell_price` first and `zero_streak` second — the Tweedie objective directly
  rewards suppressing predictions when price is low and the item has been inactive, aligning
  gain with the structural zero-mass property of the compound Poisson-Gamma distribution.
- **Gaussian** ranks `sales_rolling_mean_7` first and `sales_rolling_mean_14` second — the L2
  objective is driven purely by the conditional mean, so recent demand history dominates.
- `sales_rolling_mean_14` appears in the top 3 for both models, confirming the 2-week window
  as the most information-dense rolling summary regardless of objective.
- `days_since_last_sale` ranks fourth for Tweedie but does not appear in Gaussian's top 5,
  consistent with Tweedie needing to model the zero-demand state explicitly.

In [ ]:
top_n = 20
fi_tw = models["tweedie"].feature_importance.head(top_n).reset_index()
fi_ga = models["gaussian"].feature_importance.head(top_n).reset_index()
fi_tw.columns = fi_ga.columns = ["Feature", "Gain"]
fi_tw["Model"] = "Tweedie"
fi_ga["Model"] = "Gaussian"

# Normalise gain to 0-100 for fair comparison across models
for df_fi in [fi_tw, fi_ga]:
    df_fi["Gain_norm"] = df_fi["Gain"] / df_fi["Gain"].sum() * 100

# Union of top features from both models
top_features = list(dict.fromkeys(fi_tw["Feature"].tolist() + fi_ga["Feature"].tolist()))

fi_all = pd.concat([fi_tw, fi_ga]).reset_index(drop=True)
fi_all = fi_all[fi_all["Feature"].isin(top_features)]

fig = px.bar(
    fi_all,
    x="Gain_norm", y="Feature", color="Model",
    facet_col="Model", orientation="h",
    title="Feature Importance Comparison (normalised gain, top 20)",
    labels={"Gain_norm": "Normalised gain (%)", "Feature": ""},
    color_discrete_map={"Tweedie": "#00CC96", "Gaussian": "#FFA15A"},
    height=560,
)
fig.update_layout(showlegend=False)
fig.write_html(str(FIGS_DIR / "feature_importance_comparison.html"), include_plotlyjs="cdn")
fig.show()

## 9. SHAP Feature Impact Analysis

SHAP (SHapley Additive exPlanations) provides theoretically grounded, per-prediction
feature attributions based on cooperative game theory. Unlike LightGBM's gain-based
importance (which measures aggregate split quality), SHAP values show the exact marginal
contribution of each feature to each individual prediction — with sign and magnitude.

Three views:
- **Bar (mean |SHAP|):** Global importance ranking for each model — direction-aware equivalent of gain
- **Beeswarm:** Distribution of SHAP values per feature across all sampled predictions.
  Color = feature value (red = high, blue = low). Width = density of predictions at that SHAP value.
- **Dependence plot (sales_lag_7):** How last week's sales influence the prediction differently
  for each model and each category. Reveals where Tweedie and Gaussian diverge structurally.

In [ ]:
import shap
from retail_forecast.features import FEATURE_COLS

N_SHAP = 5000
shap_sample = test_df.sample(n=N_SHAP, random_state=42)

# Keep original dtypes including categoricals — LightGBM pred_contrib requires
# the same categorical encoding as training
X_shap = shap_sample[FEATURE_COLS].copy()

shap_values = {}
for key in ["tweedie", "gaussian"]:
    explainer = shap.TreeExplainer(models[key]._booster)
    shap_values[key] = explainer.shap_values(X_shap)
    print(f"SHAP computed for {key}: shape {shap_values[key].shape}")

In [ ]:
# Side-by-side: bar (mean |SHAP|) and beeswarm for both models
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for row, key in enumerate(["tweedie", "gaussian"]):
    plt.sca(axes[row, 0])
    shap.summary_plot(
        shap_values[key], X_shap,
        plot_type="bar", max_display=15, show=False,
        color="#00CC96" if key == "tweedie" else "#FFA15A",
    )
    axes[row, 0].set_title(f"{key.capitalize()} — Mean |SHAP|")

    plt.sca(axes[row, 1])
    shap.summary_plot(
        shap_values[key], X_shap,
        max_display=15, show=False,
    )
    axes[row, 1].set_title(f"{key.capitalize()} — SHAP Beeswarm")

plt.suptitle("SHAP Feature Impact: Tweedie vs Gaussian", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(str(FIGS_DIR / "shap_summary.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nTop 5 features by mean |SHAP| (Tweedie):")
mean_shap_tw = np.abs(shap_values["tweedie"]).mean(axis=0)
top5 = pd.Series(mean_shap_tw, index=X_shap.columns).sort_values(ascending=False).head(5)
print(top5.round(4).to_string())

print("\nTop 5 features by mean |SHAP| (Gaussian):")
mean_shap_ga = np.abs(shap_values["gaussian"]).mean(axis=0)
top5g = pd.Series(mean_shap_ga, index=X_shap.columns).sort_values(ascending=False).head(5)
print(top5g.round(4).to_string())

In [ ]:
# SHAP dependence plot: how sales_lag_7 drives predictions, split by cat_id
top_feature = "sales_lag_7"
feat_idx = list(X_shap.columns).index(top_feature)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
cat_codes = shap_sample["cat_id"].cat.codes if hasattr(shap_sample["cat_id"], "cat") else shap_sample["cat_id"]
cat_names = shap_sample["cat_id"].astype(str).values

for ax, cat in zip(axes, ["FOODS", "HOBBIES", "HOUSEHOLD"]):
    mask = cat_names == cat
    ax.scatter(
        X_shap.loc[mask, top_feature],
        shap_values["tweedie"][mask, feat_idx],
        alpha=0.3, s=8, color=CAT_COLORS[cat], label="Tweedie",
    )
    ax.scatter(
        X_shap.loc[mask, top_feature],
        shap_values["gaussian"][mask, feat_idx],
        alpha=0.2, s=8, color="#FFA15A", marker="x", label="Gaussian",
    )
    ax.axhline(0, color="black", linewidth=0.7, linestyle="--")
    ax.set_title(cat)
    ax.set_xlabel("sales_lag_7 (units sold 7 days ago)")
    if ax == axes[0]:
        ax.set_ylabel("SHAP value")
    ax.legend(fontsize=8, markerscale=2)

plt.suptitle(
    "SHAP Dependence Plot — sales_lag_7\n"
    "How last week's sales influence each model's prediction",
    y=1.02,
)
plt.tight_layout()
plt.savefig(str(FIGS_DIR / "shap_dependence_lag7.png"), dpi=120, bbox_inches="tight")
plt.show()

## 9. Save Models

Persist both models to `outputs/models/` for use in the Streamlit dashboard and `rf-predict` CLI.
Both models are retrained on train+val here to maximise in-sample coverage for production inference.

In [ ]:
# Retrain on train+val for final saved models
final_train = feat[feat["date"] <= val_end]

MODELS_DIR.mkdir(parents=True, exist_ok=True)
for ModelClass, key in [(LGBMTweedie, "tweedie"), (LGBMGaussian, "gaussian")]:
    print(f"Retraining {key} on train+val...")
    m = ModelClass(num_boost_round=1000, early_stopping_rounds=50)
    m.fit(final_train, test_df)
    m.save(MODELS_DIR / f"lgbm_{key}.pkl")
    print(f"  Saved -> {MODELS_DIR / f'lgbm_{key}.pkl'}")

## 11. Conclusions

### Overall test-set metrics

| Model    | RMSE   | MAE    | WMAPE  |
|----------|--------|--------|--------|
| Tweedie  | 1.9632 | 0.9989 | 0.6868 |
| Gaussian | 1.9744 | 1.0111 | 0.6952 |

Tweedie improvement: -0.57% RMSE, -1.21% MAE, -1.21% WMAPE.

### WRMSSE by aggregation level

| Level   | Tweedie | Gaussian | Gain    |
|---------|---------|----------|---------|
| item    | 0.9477  | 0.9537   | +0.0060 |
| dept    | 0.4567  | 0.5322   | +0.0755 |
| cat     | 0.4063  | 0.4793   | +0.0730 |
| total   | 0.3394  | 0.4185   | +0.0791 |

Both models beat the naive baseline (WRMSSE = 1.0) at every level.
Tweedie's advantage grows with aggregation — at total store level the gap is 0.079 vs 0.006 at item level.

### Summary of findings

| Finding | Evidence |
|---------|----------|
| Tweedie outperforms Gaussian overall | Section 2: -1.21% WMAPE, -0.57% RMSE |
| Advantage largest in HOBBIES (most intermittent, ~72% zeros) | Section 3 & 7 |
| Tweedie gain compounds across hierarchy — largest at total level | Section 4 WRMSSE table |
| Gaussian over-predicts zero-demand days (mean 0.5970 vs 0.5759) | Section 5 |
| Gaussian residuals skewed negative — systematic positive bias | Section 6 histograms |
| Higher zero-inflation → bigger Tweedie gain (HOBBIES +0.79% vs FOODS +0.51%) | Section 7 scatter |
| Tweedie's top SHAP features: sell_price (#1), zero_streak (#2), rolling_mean_14 (#3) | Section 8 & 9 |
| Gaussian's top SHAP features: rolling_mean_7 (#1), rolling_mean_14 (#2), lag_1 (#3) | Section 8 & 9 |

### Why Tweedie works

The Tweedie distribution with variance power `1 < p < 2` is a compound Poisson-Gamma:
it naturally places a **point mass at zero** while modelling the positive sales tail with a Gamma.
Gaussian L2 loss has no such mechanism — it pulls predictions toward the conditional mean
even when the true value is zero, introducing a systematic positive bias on zero-demand days.
SHAP analysis confirms this structurally: Tweedie assigns the highest marginal importance to
`sell_price` and `zero_streak` — features that signal when demand is likely zero —
while Gaussian weights recent rolling averages more heavily, reflecting its symmetric
squared-error objective with no zero-mass mechanism.

### When the advantage is negligible

For FOODS (least intermittent, ~56% zeros), the models are nearly equivalent.
As zero-inflation decreases, the structural Tweedie advantage shrinks and standard L2 regression
becomes a viable substitute. The choice of objective matters most for SKUs with >65% zero days.